In [27]:
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import r2_score
import numpy as np


In [2]:
df = pd.read_csv("dataset_combined.csv")

In [3]:
len(df)

124425

In [4]:
df.head(5)

,class,region,wave_2002.46,wave_2001.50,wave_2000.54,wave_1999.58,wave_1998.62,wave_1997.66,wave_1996.70,wave_1995.74,...,wave_933.84,wave_932.67,wave_931.50,wave_930.32,wave_929.15,wave_927.98,wave_926.80,source_file,X,Y
0,endo,cortex,10178.839844,10165.766602,9905.018555,9887.439453,10146.862305,10076.990234,10103.795898,10047.018555,...,8520.928711,8504.200195,8459.643555,8528.562500,8428.372070,8516.528320,8544.746094,cortex_endo_1group_633nm_center1500_obj100_pow...,8315.753404,13149.765
1,endo,cortex,8581.288086,8638.810547,8600.557617,8402.886719,8733.147461,8773.278320,8492.099609,8618.445313,...,6477.447754,6480.410156,6346.338379,6430.672852,6350.172852,6303.942871,6360.422852,cortex_endo_1group_633nm_center1500_obj100_pow...,8317.753404,13149.765
2,endo,cortex,9784.027344,9825.862305,9821.366211,9712.325195,9629.458008,9617.164063,9691.076172,9864.203125,...,7805.496094,7752.506348,7866.547852,7751.487305,7670.717773,7921.655273,7821.629883,cortex_endo_1group_633nm_center1500_obj100_pow...,8319.753404,13149.765
3,endo,cortex,10320.031250,10461.222656,10365.109375,10535.625000,10562.352539,10356.543945,10393.744141,10652.920898,...,8771.543945,8848.993164,8930.694336,8686.974609,8734.430664,8784.007813,8869.933594,cortex_endo_1group_633nm_center1500_obj100_pow...,8321.753404,13149.765
4,endo,cortex,10223.289063,10265.124023,10192.575195,10159.259766,10431.695313,10241.587891,10383.295898,10297.736328,...,8349.567383,8525.615234,8418.961914,8415.105469,8469.037109,8640.638672,8634.600586,cortex_endo_1group_633nm_center1500_obj100_pow...,8323.753404,13149.765


## ⚠️ Проверка на data leakage (утечку по группам)

При случайном split по строкам строки из **одного source_file** попадают и в train, и в test. Спектры из одного файла (одного образца) почти идентичны — модель "подглядывает" в тесте. Ниже — диагностика.

In [32]:
# Диагностика: сколько source_file попадают и в train, и в test?
from sklearn.model_selection import train_test_split

x_tmp = df.drop(['class','source_file'], axis=1)
y_tmp = df['class']
x_tr, x_te, y_tr, y_te = train_test_split(x_tmp, y_tmp, test_size=0.2, random_state=42, shuffle=True)

train_files = set(df.loc[x_tr.index, 'source_file'])
test_files = set(df.loc[x_te.index, 'source_file'])
overlap = train_files & test_files

print(f"Уникальных source_file: {df['source_file'].nunique()}")
print(f"Файлов в train: {len(train_files)}, в test: {len(test_files)}")
print(f"Файлов в ОБОИХ (утечка!): {len(overlap)} ({100*len(overlap)/len(test_files):.1f}% теста)")

Уникальных source_file: 237
Файлов в train: 237, в test: 237
Файлов в ОБОИХ (утечка!): 237 (100.0% теста)


In [33]:
from sklearn.model_selection import GroupShuffleSplit

x = df.drop(['class','source_file'], axis=1)
y = df['class']
groups = df['source_file']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x, y, groups))

x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [34]:
# Проверка: после split по группам утечки быть не должно
train_files_new = set(df.loc[train_idx, 'source_file'])
test_files_new = set(df.loc[test_idx, 'source_file'])
print("Файлов в train:", len(train_files_new), "| в test:", len(test_files_new))
print("Пересечение (должно быть 0):", len(train_files_new & test_files_new))

Файлов в train: 189 | в test: 48
Пересечение (должно быть 0): 0


In [35]:


model = CatBoostClassifier(
    cat_features=['region']
)

In [ ]:
model.fit(x_train,y_train)

Learning rate set to 0.100035
0:	learn: 1.0197998	total: 1s	remaining: 16m 43s
1:	learn: 0.9366574	total: 2s	remaining: 16m 39s
2:	learn: 0.8675259	total: 2.95s	remaining: 16m 21s


In [15]:
y_pred = model.predict(x_test)

In [25]:
def prepare_classes_to_nums(y):
    res = []
    for ob in y:
        if ob == "exo":
            res.append(0)
        elif ob == "endo":
            res.append(1)
        else:
            res.append(2)
    return np.array(res)

In [28]:
y_pred = prepare_classes_to_nums(y_pred)

In [29]:
y_test = prepare_classes_to_nums(y_test)

In [31]:
print(r2_score(y_pred,y_test))
print(f1_score(y_pred,y_test,average='micro'))

1.0
1.0
